# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [1]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [2]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

# 模型参数：w（斜率）、b（截距）、c（二次项系数）→ y_pred = w*x + b + c*x²
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
c = torch.tensor(0.0, requires_grad=True)

lr = 0.01
epochs = 2000

for epoch in range(epochs):
    # 前向传播
    y_pred = w * x + b + c * x ** 2
    loss = ((y_pred - y) ** 2).mean()

    # 反向传播
    loss.backward()

    # 更新参数
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        c -= lr * c.grad

    # 清零梯度
    w.grad.zero_()
    b.grad.zero_()
    c.grad.zero_()

# 最终结果展示
y_pred_final = w * x + b + c * x ** 2
mae = (y_pred_final - y).abs().mean()

print(f"学习到的参数：w = {w.item():.4f}, b = {b.item():.4f}, c = {c.item():.4f}")
print(f"每个样本的预测值：{y_pred_final.tolist()}")
print(f"最终平均绝对误差（MAE）：{mae.item():.6f}")

学习到的参数：w = 1.5000, b = 2.0000, c = -0.5000
每个样本的预测值：[-2.9999935626983643, -1.3749990463256836, -3.0100345611572266e-06, 1.124994158744812, 1.9999927282333374, 2.624992609024048, 2.9999935626983643, 3.1249961853027344, 3.0]
最终平均绝对误差（MAE）：0.000005


### 我的观察与说明

请在这里填写你的回答。

In [4]:
模型从线性扩展为二次多项式：作业01中模型为 y_pred = w * x（仅1个参数w），本次改为 y_pred = w*x + b + c*x²，新增截距b和二次项系数c共3个可学习参数。

数据从单样本变为批量张量：作业01使用单个x、y值，本次使用包含9个样本的torch.tensor，前向传播和损失计算均基于整个批次。

损失函数改用均值：作业01计算单样本平方误差，本次使用 ((y_pred - y)**2).mean() 计算批量均方误差（MSE）。

梯度清零方式调整：三个参数分别调用 .grad.zero_() 清零，避免梯度累积。

评估指标增加MAE：除loss外，额外计算平均绝对误差 (y_pred - y).abs().mean()，更直观反映预测偏差大小。

如何判断这次训练有效
Loss整体呈下降趋势并趋于稳定：不要求每轮都降，但宏观上应从较大值收敛到接近0的小值，说明梯度下降方向正确且已到达极小点附近。

最终MAE足够小：如MAE < 0.01，说明预测值与真实标签的平均偏差极小，模型拟合良好。

每个样本的预测值与真实值逐一对应接近：而非仅部分样本拟合好、其余偏差大，表明模型学到了全局规律而非过拟合个别点。

学到的参数代入公式能复现全部y值：将最终的w、b、c代入 y = w*x + b + c*x²，对9个x都能算出对应的y，验证了参数的正确性。

参数不再大幅跳动：训练后期w、b、c的变化幅度逐渐减小至几乎不变，说明已收敛。

SyntaxError: invalid character '：' (U+FF1A) (427755761.py, line 1)

## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。